# 10 - Matriz completa de experimentos
En este notebook ejecutamos la matriz completa de experimentos: todos los conjuntos de variables (`config.FEATURE_SETS`, 4) × todos los perfiles (`config.PROFILES`, 4) × los modelos de `config.MATRIX_MODELS` (5), en las 17 particiones. Son 17 × 4 × 4 × 5 = **1.360 filas**, una por modelo y pliegue.

Permite comparar los subconjuntos con el conjunto completo, por lo que sirve de base para contrastar las hipótesis de investigación.

## Carga de datos y conjuntos de variables

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import FEATURE_SETS, MATRIX_MODELS, PROCESSED_DATA_DIR, PROFILES, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import summarize_folds
from urban_accessibility_ml.modeling.experiments import run_experiments, save_results
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style
from urban_accessibility_ml.preprocessing.features import dimensions_for
from urban_accessibility_ml.preprocessing.split import build_all_splits

RESULTS_FILE = REPORTS_DIR / "results" / "experiment-matrix-default.csv"
STRATEGIES = ["within_city", "mixed", "cross_city"]
# Refutation criterion of the thesis: "clearly inferior" = below 80 % of the F1 macro of the complete set
CLEARLY_INFERIOR_RATIO = 0.8
FEATURE_SET_COLORS = dict(zip(FEATURE_SETS, SERIES))

use_style()

df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv")
pd.Series({feature_set: dimensions_for(feature_set) for feature_set in FEATURE_SETS}, name="dimensiones")

Como nota, con el umbral actual (`config.THRESHOLD`, que es 50%), el conjunto `combinado` tiene las mismas dimensiones que `calle`, así que sus resultados serán idénticos y la hipótesis H3 no se puede contrastar.

## Particiones

In [ ]:
splits = build_all_splits(df)
len(splits)

## Entrenamiento y evaluación
Ejecutamos `run_experiments` con los valores por defecto.

In [ ]:
results = run_experiments(df, splits)
save_results(results, RESULTS_FILE)
print(len(results), "filas =", len(splits), "particiones x", len(FEATURE_SETS), "conjuntos x", len(PROFILES), "perfiles x", len(MATRIX_MODELS), "modelos")
results.head()

## Resumen por conjunto, perfil y modelo
Calculamos la media y desviación típica entre pliegues, separadas por estrategia (`within_city`, `cross_city` y `mixed`).

In [ ]:
summary = summarize_folds(results, ["strategy", "feature_set", "profile", "model"])
summary.round(3)

## Matriz agregada de F1 macro medio

Vamos a utilizar la F1 macro como comparador en una matriz agregada.

In [ ]:
summary.pivot(index=["strategy", "feature_set", "profile"], columns="model", values="f1_macro_mean").round(3)

## F1 macro por cada conjunto de variables
Vamos a mostrar un gráfico por cada modelo, sin el `dummy`, que solo sirve de línea base, para cada conjunto de variables, estrategia y perfil. Cada columna es una estrategia, con un perfil por serie. No elegimos ninguno según el test.

La desviación típica entre pliegues se muestra con una barra vertical negra.

La línea discontinua marca el 80 % de la F1 macro del conjunto `completo` con el mismo modelo, estrategia y perfil. Si la barra de uno de los subconjuntos queda por debajo de esa línea, se podrían descartar algunas de las hipótesis de la memoria.

In [ ]:
PROFILE_COLORS = dict(zip(PROFILES, SERIES))


def plot_model(model: str) -> None:
    """Dibuja y guarda la F1 macro por conjunto, estrategia y perfil de un modelo."""
    x = np.arange(len(FEATURE_SETS))
    width = 0.2

    fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(13, 4.6), sharey=True)
    for ax, strategy in zip(axes, STRATEGIES):
        data = summary[(summary["model"] == model) & (summary["strategy"] == strategy)]
        for i, profile in enumerate(PROFILES):
            rows = data[data["profile"] == profile].set_index("feature_set").loc[FEATURE_SETS]
            positions = x + (i - (len(PROFILES) - 1) / 2) * width
            ax.bar(
                positions,
                rows["f1_macro_mean"],
                width,
                yerr=rows["f1_macro_std"].fillna(0),
                color=PROFILE_COLORS[profile],
                label=profile,
                error_kw={"linewidth": 1},
            )
            # 80 % de la F1 macro de completo del mismo modelo, estrategia y perfil
            ax.hlines(
                np.full(
                    len(FEATURE_SETS),
                    CLEARLY_INFERIOR_RATIO * rows.loc["completo", "f1_macro_mean"],
                ),
                positions - width / 2,
                positions + width / 2,
                colors="black",
                linestyles="--",
                linewidth=1,
                label="80 % de completo" if i == 0 else None,
            )
        ax.set_title(strategy)
        ax.set_xticks(x, FEATURE_SETS)
        ax.set_ylim(0, 1)
    axes[0].set_ylabel(f"F1 macro ({model})")
    handles, names = axes[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="lower center", ncol=len(names), frameon=False)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save_figure(f"f1_macro_por_conjunto_{model}", "0.10-matriz-experimentos", fig=fig)
    plt.show()


def cases_below(model: str) -> pd.DataFrame:
    """Casos del modelo con la F1 macro por debajo del 80 % de la de completo."""
    data = summary[summary["model"] == model]
    reference = data[data["feature_set"] == "completo"][["strategy", "profile", "f1_macro_mean"]]
    subsets = data[data["feature_set"] != "completo"].merge(
        reference, on=["strategy", "profile"], suffixes=("", "_completo")
    )
    subsets["80 % de completo"] = CLEARLY_INFERIOR_RATIO * subsets["f1_macro_mean_completo"]
    below = subsets[subsets["f1_macro_mean"] < subsets["80 % de completo"]]
    columns = ["strategy", "feature_set", "profile", "f1_macro_mean", "80 % de completo"]
    return (
        below[columns]
        .rename(columns={"f1_macro_mean": "F1 macro"})
        .round(3)
        .reset_index(drop=True)
    )

In [ ]:
plot_model("logistic_regression")

In [ ]:
cases_below("logistic_regression")

Las barras verticales negras indican la desviación típica.

Con la regresión logística la F1 macro va de 0,61 a 0,80 en `cross_city`, de 0,63 a 0,87 en `mixed` y de 0,64 a 0,92 en `within_city`.

Hay cuatro casos por debajo de la línea discontinua del 80%, y todos ocurren en el conjunto cartográfico y aéreo. Son sensorial en `cross_city` (0,612 frente a 0,630), global y sensorial en `mixed` (0,674 frente a 0,694 y 0,627 frente a 0,630) y global en `within_city` (0,644 frente a 0,706).

In [ ]:
plot_model("random_forest")

In [ ]:
cases_below("random_forest")

Las barras verticales negras indican la desviación típica.

Vemos que la estrategia de transferencia entre ciudades (`cross_city`) es la que obtiene peores resultados en términos absolutos, con F1 macro entre 0,57 y 0,72, frente a valores de 0,75 a 0,99 en las otras dos estrategias. Lo que parece indicar que el modelo generaliza peor para la transferencia entre ciudades.

Las otras dos estrategias parecen más optimistas, a excepción de la estrategia mixta para el conjunto de dimensiones cartográfico y aéreo, en cognitiva y global. En los dos casos la barra parece quedar por debajo de la línea discontinua del 80% respecto al conjunto completo.

En cognitiva la F1 macro es de 0,746 y la línea está en 0,776 (el 80 % de `completo`), mientras que en global es de 0,769 frente a 0,773, así que ahí la diferencia es mucho menor, con un 0,010 de desviación.

In [ ]:
plot_model("xgboost_balanced")

In [ ]:
cases_below("xgboost_balanced")

En XGBoost la transferencia entre ciudades (`cross_city`) obtiene resultados mucho peores, con una F1 macro entre 0,33 y 0,57, incluso con el conjunto `completo`, que queda entre 0,33 y 0,50.

Solo hay un caso por debajo de la línea discontinua del 80%, el de la estrategia mixta con el conjunto cartográfico y aéreo en cognitiva (0,746 frente a 0,761).

In [ ]:
plot_model("lightgbm")

In [ ]:
cases_below("lightgbm")

LightGBM se parece a `random_forest` en `mixed` y `within_city`, con valores entre 0,75 y 0,98. En `cross_city` la F1 macro va de 0,45 a 0,80, un rango más amplio que el de `random_forest` (0,57 a 0,72).

Hay tres casos distintos por debajo de la línea discontinua del 80%, aunque la tabla muestra cuatro filas porque `calle` y `combinado` dan el mismo resultado al tener las mismas dimensiones.

Uno es cognitiva en la estrategia mixta con el conjunto cartográfico y aéreo (0,746 frente a 0,773).
Otro es global en `cross_city` con `calle` y con `combinado` (0,592 frente a 0,637 en los dos).
El tercero es funcional en `cross_city` con el conjunto cartográfico y aéreo (0,470 frente a 0,484).

In [ ]:
wide = summary.pivot(
    index=["strategy", "profile", "model"], columns="feature_set", values="f1_macro_mean"
)
ratio = wide.drop(columns="completo").div(wide["completo"], axis=0)
below = (ratio < CLEARLY_INFERIOR_RATIO).groupby("model").sum()
below

Como podemos ver, el conjunto cartográfico y aéreo es el que más veces queda por debajo del 80% de `completo`, con 4 casos de 12 en la regresión logística, 2 en `random_forest`, 2 en LightGBM y 1 en XGBoost.

Los conjuntos `calle` y `combinado` solo aparecen una vez, en LightGBM, y es el mismo caso porque tienen las mismas dimensiones.

Por tanto, el número de casos depende del modelo, y no podemos hablar de un resultado igual en los cuatro. Además, esta tabla usa los valores por defecto, no los ajustados.

## Comprobación: `combinado` frente a `calle`

Vamos a comprobar a partir de los datos que con el umbral utilizado, el conjunto combinado es igual al que es obtenible a partir de las imágenes a pie de calle.

In [ ]:
# `wide` se calculó antes, en la tabla de casos por debajo del 80 %
print("combinado == calle en todos los casos:", bool((wide["combinado"] == wide["calle"]).all()))